# SynGlue API Usage Guide

This notebook demonstrates how to use the SynGlue Python client for both design and screening workflows. **Important notes and workflow highlights are included.**

## 1. Setup and Import
Install and import the SynGlue client and required libraries.

In [ ]:
!pip install synglue
from synglue import SynGlue
import pandas as pd
import matplotlib.pyplot as plt

## 2. Health Check and Client Initialization
Check the API health and initialize the SynGlue client.

In [ ]:
client = SynGlue()
health = client.health_check()
print("API Health:", health)

---
# DESIGN WORKFLOW
---

## 3. Submit a Design Job
Submit a design job with a single target (no CSV support).

In [ ]:
design_result = client.submit_design(target="EGFR", threshold=80)
print("Design job response:", design_result)

### Check Design Job Status
Monitor the status of your design job.

In [ ]:
design_job_id = design_result.get("job_id")
design_status = client.design_status(job_id=design_job_id)
print("Design job status:", design_status)

### Download Design Results
Download results after the job is complete.

In [ ]:
# Uncomment when job is complete
client.download_design(job_id=design_job_id, out_path="design_results.zip")

## Unzip and Display Results for Design
 Jobs
Typical design results zip structure:
- Exit_Vectors.png (image)
- latest_run/
    - ADMET_Predictions_Top_20.csv
    - Final_Predicted_PROTACs.csv
    - Final_Top_3_Predicted_PROTACs.png
    - results/
        - scaffold_memory.csv

The following cell will extract and display all images and CSVs from this structure.

In [ ]:
import zipfile
import pandas as pd
import os

# Path to your downloaded design results zip file
zip_path = "content/design_job_results.zip"  # <-- update this path

# Create a temporary directory to extract
extract_dir = "design_results_extracted"
os.makedirs(extract_dir, exist_ok=True)

# Unzip the results
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# List all files extracted
print("Extracted files:")
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(os.path.join(root, file))

# Display the head and info of key CSVs
# Update these paths if your structure is different
latest_run_dir = os.path.join(extract_dir, "latest_run")
results_dir = os.path.join(latest_run_dir, "results")

# Example: Display Final_Predicted_PROTACs.csv
final_protacs = os.path.join(latest_run_dir, "Final_Predicted_PROTACs.csv")
if os.path.exists(final_protacs):
    print("\nFinal_Predicted_PROTACs.csv:")
    df = pd.read_csv(final_protacs)
    display(df.head())
    print(df.info())
else:
    print("Final_Predicted_PROTACs.csv not found.")

# Example: Display ADMET_Predictions_Top_20.csv
admet_csv = os.path.join(latest_run_dir, "ADMET_Predictions_Top_20.csv")
if os.path.exists(admet_csv):
    print("\nADMET_Predictions_Top_20.csv:")
    df = pd.read_csv(admet_csv)
    display(df.head())
    print(df.info())
else:
    print("ADMET_Predictions_Top_20.csv not found.")

# Example: Display scaffold_memory.csv (in results/)
scaffold_csv = os.path.join(results_dir, "scaffold_memory.csv")
if os.path.exists(scaffold_csv):
    print("\nscaffold_memory.csv:")
    df = pd.read_csv(scaffold_csv)
    display(df.head())
    print(df.info())
else:
    print("scaffold_memory.csv not found.")

---
# SCREEN WORKFLOW (List of Molecules)
---

## 4. Submit a Screen Job (List)
Submit a screen job with a list of molecules.

In [ ]:
molecules = [
    {"name": "Aspirin", "smiles": "CC(=O)Oc1ccccc1C(=O)O"},
    {"name": "Imatinib", "smiles": "CC1=CC=CC=C1"}
]
screen_result = client.submit_screen(molecules=molecules)
print("Screen job response:", screen_result)

### Check Screen Job Status (List)
Monitor the status of your screen job.

In [ ]:
screen_job_id = screen_result.get("job_id")
screen_status = client.screen_status(job_id=screen_job_id)
print("Screen job status:", screen_status)

### Download Screen Results (List)
Download results after the job is complete.

## Unzip and Display Results for Screen Jobs

This section demonstrates how to extract and display the results from a SynGlue screen job (hybrid mapping), regardless of whether you submitted molecules as a list or via CSV upload.

The results zip will always contain:
- `query.csv`: Your input molecules
- `Hybrid_Mapping_Results.csv`: The mapping results

The following code will extract and display both files.

In [ ]:
import zipfile
import pandas as pd
import os

# Path to your downloaded screen results zip file
zip_path = "content/screen_job_results.zip"  # <-- update this path

# Create a temporary directory to extract
extract_dir = "screen_results_extracted"
os.makedirs(extract_dir, exist_ok=True)

# Unzip the results
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Display the input query
query_csv = os.path.join(extract_dir, "query.csv")
if os.path.exists(query_csv):
    print("Input Query (query.csv):")
    display(pd.read_csv(query_csv))
else:
    print("query.csv not found in the zip.")

# Display the mapping results
results_csv = os.path.join(extract_dir, "Hybrid_Mapping_Results.csv")
if os.path.exists(results_csv):
    print("Hybrid Mapping Results (Hybrid_Mapping_Results.csv):")
    display(pd.read_csv(results_csv))
else:
    print("Hybrid_Mapping_Results.csv not found in the zip.")

---
# SCREEN WORKFLOW (CSV)
---

## 5. Submit a Screen Job (CSV)
**Only the screening workflow supports CSV input.**

Submit a screen job by uploading a CSV file with columns `name` and `smiles`.

In [ ]:
csv_path = "../data/grover_e3.csv"  # Update with your CSV path
screen_csv_result = client.submit_screen_csv(csv_path)
print("Screening job submitted. Job ID:", screen_csv_result["job_id"])

### Check Screen Job Status (CSV)
Monitor the status of your CSV-based screen job.

In [ ]:
screen_csv_job_id = screen_csv_result.get("job_id")
screen_csv_status = client.screen_status(job_id=screen_csv_job_id)
print("Screening job status:", screen_csv_status)

### Download Screen Results (CSV)
Download results after the job is complete.

In [ ]:
# Uncomment when job is complete
# client.download_screen(job_id=screen_csv_job_id, out_path="screen_results_csv.zip")